In [4]:
!pip install datasets
from datasets import load_dataset

In [5]:
# load data
data = load_dataset("cornell-movie-review-data/rotten_tomatoes")
data # 这里显示数据集有train、validation、test三个划分

README.md:   0%|          | 0.00/7.46k [00:00<?, ?B/s]

train.parquet: reconstructing file:   0%|          |  0.00B /  699kB            

train.parquet: downloading bytes:           |  0.00B            

validation.parquet: reconstructing file:   0%|          |  0.00B / 90.0kB            

validation.parquet: downloading bytes:           |  0.00B            

...

test.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/8530 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1066 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1066 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 8530
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
})

In [6]:
data["train"][0, -1]

{'text': ['the rock is destined to be the 21st century\'s new " conan " and that he\'s going to make a splash even greater than arnold schwarzenegger , jean-claud van damme or steven segal .',
  'things really get weird , though not particularly scary : the movie is all portent and no content .'],
 'label': [1, 0]}

In [1]:
from transformers import pipeline

model_path = "cardiffnlp/twitter-roberta-base-sentiment-latest"

pipe = pipeline(
    model=model_path,
    tokenizer=model_path,
    return_all_scores=True, # 返回所有类别的分数
    device="cuda:0"
)

config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  501MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  501MB            

model.safetensors: downloading bytes:           |  0.00B            

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [26]:
output = pipe(
    data["test"][0]["text"],
    top_k=None
)

print(output)
# 这部分代码是debug用，排查单次输出的结果
# 也正是通过这部分代码的打印发现topk=None返回的顺序不是严格按照0是negative，2是positive来的，所以后续的计算出错了,是按照分数大小顺序来的

[{'label': 'positive', 'score': 0.9546052813529968}, {'label': 'neutral', 'score': 0.040233541280031204}, {'label': 'negative', 'score': 0.00516123790293932}]


In [22]:
import numpy as np
from tqdm import tqdm
from transformers.pipelines.pt_utils import KeyDataset
## 创建一个惰性迭代器（Lazy Iterator），能够高效地将Pipeline应用到数据集的某个指定列上，而不会一次性将所有数据加载到内存中。

y_pred = []
# 注意这个地方目前模型默认topk=1，所以只返回一个值，是字典而非字典数组，所以指定topk=None符合当前的写法
for output in tqdm(pipe(KeyDataset(data["test"], "text"), top_k=None),
total=len(data["test"])):
  score_dict = {
        item["label"]: item["score"]
        for item in output
    }
  negative_score = score_dict["negative"]
  positive_score = score_dict["positive"]
  assignment = np.argmax(
        [negative_score, positive_score]
    )
  y_pred.append(assignment)
# 这里必须根据先根据标签分类再去比较分数，因为不是严格按照第一个是negative这个顺序返回的

100%|██████████| 1066/1066 [00:10<00:00, 98.38it/s] 


In [23]:
y_pred

[np.int64(1),
 np.int64(1),
 np.int64(0),
 np.int64(1),
 np.int64(1),
 np.int64(0),
 np.int64(1),
 np.int64(1),
 np.int64(1),
 np.int64(1),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(1),
 np.int64(0),
 np.int64(1),
 np.int64(1),
 np.int64(1),
 np.int64(1),
 np.int64(1),
 np.int64(1),
 np.int64(0),
 np.int64(0),
 np.int64(1),
 np.int64(1),
 np.int64(0),
 np.int64(1),
 np.int64(1),
 np.int64(0),
 np.int64(1),
 np.int64(1),
 np.int64(0),
 np.int64(1),
 np.int64(1),
 np.int64(1),
 np.int64(1),
 np.int64(0),
 np.int64(1),
 np.int64(1),
 np.int64(1),
 np.int64(0),
 np.int64(1),
 np.int64(1),
 np.int64(0),
 np.int64(1),
 np.int64(0),
 np.int64(1),
 np.int64(1),
 np.int64(1),
 np.int64(1),
 np.int64(0),
 np.int64(1),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(1),
 np.int64(0),
 np.int64(1),
 np.int64(1),
 np.int64(0),
 np.int64(0),
 np.int64(1),
 np.int64(1),
 np.int64(0),
 np.int64(0),
 np.int64(1),
 np.int64(0),
 np.int64(1),
 np.int64(1),
 np.int64(1),
 np.int64(1),
 np.in

In [24]:
from sklearn.metrics import classification_report

def evaluate_preformance(y_true, y_pred):
  performance = classification_report(
      y_true, y_pred,
      target_names=["Negative Review", "Positive Review"],
  )
  print(performance)

In [25]:
evaluate_preformance(data["test"]["label"], y_pred)

                 precision    recall  f1-score   support

Negative Review       0.76      0.88      0.81       533
Positive Review       0.86      0.72      0.78       533

       accuracy                           0.80      1066
      macro avg       0.81      0.80      0.80      1066
   weighted avg       0.81      0.80      0.80      1066

